In [6]:
# Install PyCryptodome and PyCUDA in Colab
!pip install pycryptodome pycuda

# All dependencies
from google.colab import files
import ecdsa
from Crypto.Hash import RIPEMD160
import hashlib
import base58
import multiprocessing
import time
import random
import json
import os
import sys
import numpy as np
import pycuda.driver as cuda_drv
from pycuda.compiler import SourceModule
import math

# Verify T4 GPU availability
!nvidia-smi

# --- Configuration ---
# File containing target Bitcoin addresses (one per line)
TARGET_FILENAME = None  # Will be set after upload

# Files for persistence
BEST_KEY_FILE = 'best_key.json'
CRACKED_FILE = 'cracked.txt'
CHECKPOINT_FILE = 'progress.json'

# Search Space (Chaos Range based on 64-bit seed)
# Focuses the search between 2^58 and roughly 2^60
CHAOS_LOWER = int(2**58)  # ~2.88 × 10^17
CHAOS_UPPER = int(2**60 - 1)  # ~1.15 × 10^18 (fits uint64_t)
RANGE_SIZE = CHAOS_UPPER - CHAOS_LOWER
if RANGE_SIZE <= 0:
    raise ValueError("CHAOS_UPPER must be greater than CHAOS_LOWER")

print(f"Searching keys in decimal range: [{CHAOS_LOWER}, {CHAOS_UPPER}]")
print(f"Range size: {RANGE_SIZE} keys")
print(f"Equivalent bit range: ~{math.log2(CHAOS_LOWER):.2f} to ~{math.log2(CHAOS_UPPER):.2f} bits")

# --- Target Address Loading ---
try:
    print("Please upload the file containing target Bitcoin addresses:")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file was uploaded.")
    TARGET_FILENAME = list(uploaded.keys())[0]
    print(f"Uploaded file: {TARGET_FILENAME}")

    with open(TARGET_FILENAME, 'r') as f:
        # Use a set for fast lookups
        TARGET_ADDRESSES = set(line.strip() for line in f if line.strip() and line.strip().startswith('1'))

    if not TARGET_ADDRESSES:
        print("Error: No valid target addresses found in the file. Addresses should start with '1'. Exiting.")
        sys.exit(1)

    print(f"Loaded {len(TARGET_ADDRESSES)} target addresses into memory.")

except FileNotFoundError as e:
    print(f"Error loading target file: {e}. Please ensure the file exists and was uploaded correctly.")
    sys.exit(1)
except Exception as e:
    print(f"An unexpected error occurred during file processing: {e}")
    sys.exit(1)

# --- Secp256k1 Setup ---
G = ecdsa.SECP256k1.generator
ORDER = G.order()  # Order of the secp256k1 curve

# --- Persistence Functions ---
def save_best_key(key, bits):
    """Saves the best key found so far (globally)."""
    try:
        with open(BEST_KEY_FILE, 'w') as f:
            json.dump({'key': hex(key), 'bits': bits}, f)
    except IOError as e:
        print(f"Warning: Could not save best key file: {e}")

def load_best_key():
    """Loads the best key found previously, or initializes."""
    if os.path.exists(BEST_KEY_FILE):
        try:
            with open(BEST_KEY_FILE, 'r') as f:
                data = json.load(f)
                # Ensure the loaded key is within our target range
                key_int = int(data['key'], 16)
                key_in_range = (key_int % RANGE_SIZE) + CHAOS_LOWER
                return key_in_range, data['bits']
        except (IOError, json.JSONDecodeError, KeyError, ValueError) as e:
            print(f"Warning: Could not load or parse {BEST_KEY_FILE}: {e}. Using default seed.")

    # Default Initialization: Use lower 64 bits of your seed, mapped to range
    seed_start = int('0x19d9230af77e896adc5ea6ea404070b19db4a021586bae46ee0f061a0ab81', 16) & 0xFFFFFFFFFFFFFFFF
    initial_key = (seed_start % RANGE_SIZE) + CHAOS_LOWER
    initial_bits = 160  # Max possible bits off for RIPEMD160 hash (20 bytes * 8 bits)
    print(f"No previous best key found. Starting with initial key: {hex(initial_key)}")
    return initial_key, initial_bits

def load_checkpoint():
    """Loads checkpoint data for workers."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r') as f:
                return json.load(f)
        except (IOError, json.JSONDecodeError) as e:
            print(f"Warning: Could not load or parse checkpoint file {CHECKPOINT_FILE}: {e}. Starting fresh.")
    return {}

def save_checkpoint(worker_id, next_seed, steps, best_bits_worker):
    """Saves the progress of a specific worker."""
    data = load_checkpoint()
    data[str(worker_id)] = {'next_seed': hex(next_seed), 'steps': steps, 'best_bits': best_bits_worker}
    try:
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump(data, f, indent=4)
    except IOError as e:
        print(f"Warning: Could not save checkpoint file: {e}")

# --- Bitcoin Address Generation ---
def priv_to_p2pkh_compressed(priv_key_int):
    """Generates a compressed P2PKH Bitcoin address from a private key integer."""
    if not (1 <= priv_key_int < ORDER):
        pass  # Allow keys outside curve order if in CHAOS range

    try:
        sk = ecdsa.SigningKey.from_secret_exponent(priv_key_int, curve=ecdsa.SECP256k1)
        vk = sk.get_verifying_key()
        point = vk.pubkey.point
        x = point.x()
        y = point.y()

        # Compressed public key format
        prefix = b'\x02' if y % 2 == 0 else b'\x03'
        pub_key_bytes = prefix + x.to_bytes(32, 'big')

        # Hashing: SHA256 -> RIPEMD160
        sha = hashlib.sha256(pub_key_bytes).digest()
        rip = RIPEMD160.new(sha).digest()  # rip is 20 bytes

        # Base58Check encoding
        extended_rip = b'\x00' + rip  # Add version byte (0x00 for mainnet P2PKH)
        checksum = hashlib.sha256(hashlib.sha256(extended_rip).digest()).digest()[:4]
        address_bytes = extended_rip + checksum

        return base58.b58encode(address_bytes).decode('ascii')

    except ValueError as e:
        return None  # Indicate failure
    except Exception as e:
        return None

# --- Closeness Metric ---
def bits_off(addr, local_target_addresses):
    """Calculates the minimum Hamming distance between the RIPEMD160 hash of the generated address and targets."""
    min_diff_bits = 160  # Max possible difference for 20-byte hash
    try:
        addr_bytes_decoded = base58.b58decode(addr)
        addr_ripemd160 = addr_bytes_decoded[1:-4]  # Extract RIPEMD160 hash (bytes)
        addr_int = int.from_bytes(addr_ripemd160, 'big')

        for target_addr in local_target_addresses:
            try:
                target_bytes_decoded = base58.b58decode(target_addr)
                target_ripemd160 = target_bytes_decoded[1:-4]
                target_int = int.from_bytes(target_ripemd160, 'big')

                # Calculate XOR difference
                diff = addr_int ^ target_int
                # Count set bits (Hamming distance)
                bits = bin(diff).count('1')
                min_diff_bits = min(min_diff_bits, bits)

            except Exception:
                continue

    except Exception:
        return 160  # Return max difference if error occurs

    return min_diff_bits

# --- CUDA Kernel ---
kernel_code = """
#include <stdint.h>
#define RANGE_SIZE 0x%016llxULL // CHAOS_UPPER - CHAOS_LOWER
#define CHAOS_LOWER 0x%016llxULL // 2^58

__global__ void chaotic_step_kernel(const uint64_t *start_seed_arr, const float *current_best_bits_arr, uint64_t *output_keys, const uint64_t num_keys) {
    unsigned int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < num_keys) {
        uint64_t current_key = start_seed_arr[0] + idx;
        current_key = (current_key % RANGE_SIZE) + CHAOS_LOWER;

        // SHA-256 mimic for chaos
        uint64_t h = 0;
        for (int i = 0; i < 8; i++) {
            h += ((current_key >> (i * 8)) & 0xFFULL) * (uint64_t)(i + 31);
            h = (h << 3) | (h >> 61);  // Rotate left
        }
        h ^= current_key;

        float current_best_bits = current_best_bits_arr[0];
        int step = max(1, 1000000 / ((int)current_best_bits + 1));
        step = min(step, 1000000);

        uint64_t next_key = (current_key + (h % (uint64_t)step) + 1) % RANGE_SIZE + CHAOS_LOWER;
        output_keys[idx] = next_key;
    }
}
""" % (RANGE_SIZE, CHAOS_LOWER)

# Compile the CUDA kernel
try:
    cuda_drv.init()
    device = cuda_drv.Device(0)
    ctx = device.make_context()
    ctx.push()

    compiled_module = SourceModule(kernel_code)
    chaotic_step_kernel_gpu = compiled_module.get_function("chaotic_step_kernel")

    ctx.pop()
    print("CUDA kernel compiled successfully.")
except cuda_drv.Error as e:
    print(f"CUDA Error during initialization or compilation: {e}")
    sys.exit(1)
except Exception as e:
    print(f"Error compiling CUDA kernel: {e}")
    sys.exit(1)

# --- CUDA Worker ---
def cuda_worker(worker_id, result_queue, initial_seed, initial_best_bits, target_addrs_local, batch_size=5000000, keys_to_check_per_cycle=500000000):
    print(f"Worker {worker_id}: Starting. Initial seed: {hex(initial_seed)}, Initial best bits: {initial_best_bits}")
    ctx = None
    d_start_seed = None
    d_output_keys = None
    d_best_bits = None

    try:
        cuda_drv.init()
        device = cuda_drv.Device(0)
        ctx = device.make_context()
        ctx.push()

        checkpoint_data = load_checkpoint().get(str(worker_id), {})
        current_seed = int(checkpoint_data.get('next_seed', hex(initial_seed)), 16)
        current_seed = (current_seed % RANGE_SIZE) + CHAOS_LOWER

        worker_steps = checkpoint_data.get('steps', 0)
        best_bits_worker = min(initial_best_bits, checkpoint_data.get('best_bits', initial_best_bits))
        best_key_worker = current_seed

        print(f"Worker {worker_id}: Resuming/Starting with seed: {hex(current_seed)}, steps: {worker_steps}, best_bits: {best_bits_worker}")

        threads_per_block = 256
        blocks_per_grid = min(device.get_attribute(cuda_drv.device_attribute.MAX_GRID_DIM_X), (batch_size + threads_per_block - 1) // threads_per_block)
        effective_batch_size = blocks_per_grid * threads_per_block
        print(f"Worker {worker_id}: Using {threads_per_block} threads/block, {blocks_per_grid} blocks/grid. Effective batch size: {effective_batch_size}")

        d_start_seed = cuda_drv.mem_alloc(np.uint64().nbytes)
        d_best_bits = cuda_drv.mem_alloc(np.float32().nbytes)
        d_output_keys = cuda_drv.mem_alloc(effective_batch_size * np.uint64().nbytes)
        keys_host = np.zeros(effective_batch_size, dtype=np.uint64)

        total_keys_processed = worker_steps
        start_time_worker = time.time()

        while total_keys_processed < keys_to_check_per_cycle:
            batch_start_time = time.time()

            cuda_drv.memcpy_htod(d_start_seed, np.array([current_seed], dtype=np.uint64))
            cuda_drv.memcpy_htod(d_best_bits, np.array([best_bits_worker], dtype=np.float32))

            chaotic_step_kernel_gpu(
                d_start_seed,
                d_best_bits,
                d_output_keys,
                np.uint64(effective_batch_size),
                block=(threads_per_block, 1, 1),
                grid=(blocks_per_grid, 1)
            )

            ctx.synchronize()
            cuda_drv.memcpy_dtoh(keys_host, d_output_keys)

            batch_gpu_time = time.time() - batch_start_time

            batch_cpu_start_time = time.time()
            keys_processed_in_batch = 0
            for i in range(effective_batch_size):
                priv_key = int(keys_host[i])
                addr = priv_to_p2pkh_compressed(priv_key)

                if addr is None:
                    continue

                keys_processed_in_batch += 1
                total_keys_processed += 1

                if addr in target_addrs_local:
                    print(f"\n!!! WORKER {worker_id} FOUND A MATCH !!!\nAddress: {addr}\nPrivate Key: {hex(priv_key)}\n")
                    with open(CRACKED_FILE, 'a') as f:
                        f.write(f"Timestamp: {time.time():.0f}, Worker: {worker_id}\n")
                        f.write(f"Address: {addr}\nPrivate Key: {hex(priv_key)}\n---\n")
                    result_queue.put({'type': 'found', 'key': priv_key, 'address': addr})
                    return

                bits = bits_off(addr, target_addrs_local)
                if bits < best_bits_worker:
                    best_bits_worker = bits
                    best_key_worker = priv_key
                    print(f"\nWorker {worker_id}: New BEST (Worker Local): {bits} bits off ({addr} from key {hex(best_key_worker)})")
                    result_queue.put({'type': 'best', 'key': best_key_worker, 'bits': best_bits_worker, 'worker_id': worker_id})
                    cuda_drv.memcpy_htod(d_best_bits, np.array([best_bits_worker], dtype=np.float32))

                if total_keys_processed % 1000000 == 0:
                    next_seed_for_batch = int(keys_host[-1])
                    save_checkpoint(worker_id, next_seed_for_batch, total_keys_processed, best_bits_worker)
                    _, global_best_bits = load_best_key()
                    if global_best_bits < best_bits_worker:
                        print(f"Worker {worker_id}: Syncing best bits with global ({global_best_bits})")
                        best_bits_worker = global_best_bits
                        cuda_drv.memcpy_htod(d_best_bits, np.array([best_bits_worker], dtype=np.float32))

            batch_cpu_time = time.time() - batch_cpu_start_time
            total_batch_time = time.time() - batch_start_time
            keys_per_sec = keys_processed_in_batch / total_batch_time if total_batch_time > 0 else 0

            print(f"Worker {worker_id}: Batch done. Keys: {keys_processed_in_batch}. GPU: {batch_gpu_time:.2f}s, CPU: {batch_cpu_time:.2f}s, Total: {total_batch_time:.2f}s ({keys_per_sec:.2f} keys/s). Total Keys: {total_keys_processed}. Best bits: {best_bits_worker}")

            current_seed = int(keys_host[-1])

        print(f"Worker {worker_id}: Reached target keys for this cycle ({keys_to_check_per_cycle}). Stopping.")
        result_queue.put({'type': 'stopped', 'worker_id': worker_id})

    except cuda_drv.Error as e:
        print(f"Worker {worker_id}: CUDA Error: {e}")
        result_queue.put({'type': 'error', 'worker_id': worker_id, 'message': str(e)})
    except Exception as e:
        print(f"Worker {worker_id}: General Error: {e}")
        import traceback
        traceback.print_exc()
        result_queue.put({'type': 'error', 'worker_id': worker_id, 'message': str(e)})
    finally:
        print(f"Worker {worker_id}: Cleaning up...")
        try:
            if d_start_seed: d_start_seed.free()
            if d_output_keys: d_output_keys.free()
            if d_best_bits: d_best_bits.free()
        except cuda_drv.Error as free_err:
            print(f"Worker {worker_id}: Warning - CUDA error during memory cleanup: {free_err}")
        if ctx:
            ctx.pop()
            print(f"Worker {worker_id}: CUDA context released.")

# --- Main Process Orchestration ---
def main_coordinator(target_addrs):
    print("\n--- Starting Bitcoin Address Hunter ---")
    print(f"Targeting {len(target_addrs)} addresses.")
    print(f"Using CUDA T4 with Chaotic Search in range 2^58 to ~2^60.")
    print(f"Persistence files: {BEST_KEY_FILE}, {CRACKED_FILE}, {CHECKPOINT_FILE}")
    print("Press Ctrl+C to stop gracefully.")

    multiprocessing.set_start_method('spawn', force=True)

    result_queue = multiprocessing.Queue()
    processes = []

    num_workers = min(multiprocessing.cpu_count(), 4)
    print(f"Starting {num_workers} worker processes...")

    global_best_key_start, global_best_bits_start = load_best_key()
    print(f"Initializing workers with global best: key ~{hex(global_best_key_start)}, bits {global_best_bits_start}")

    active_workers = num_workers
    for i in range(num_workers):
        worker_initial_seed = ((global_best_key_start + i * 1000) % RANGE_SIZE) + CHAOS_LOWER
        p = multiprocessing.Process(
            target=cuda_worker,
            args=(i, result_queue, worker_initial_seed, global_best_bits_start, target_addrs),
            daemon=True
        )
        processes.append(p)
        p.start()
        time.sleep(1)

    start_time_main = time.time()
    found_key_info = None
    try:
        while active_workers > 0:
            try:
                result = result_queue.get(timeout=10.0)

                if result['type'] == 'found':
                    print("\n" + "="*20 + " MATCH FOUND! " + "="*20)
                    found_key_info = result
                    print(f"Address: {result['address']}")
                    print(f"Private Key (hex): {hex(result['key'])}")
                    print("="*54)
                    break

                elif result['type'] == 'best':
                    worker_id = result['worker_id']
                    current_bits = result['bits']
                    current_key = result['key']
                    _, global_best_bits = load_best_key()
                    if current_bits < global_best_bits:
                        print(f"\n--- Main: New GLOBAL BEST found by Worker {worker_id} ---")
                        print(f"Bits off: {current_bits}")
                        print(f"Key: {hex(current_key)}")
                        save_best_key(current_key, current_bits)

                elif result['type'] == 'stopped':
                    print(f"Main: Worker {result['worker_id']} finished its cycle.")
                    active_workers -= 1

                elif result['type'] == 'error':
                    print(f"Main: Worker {result['worker_id']} reported an error: {result['message']}")
                    active_workers -= 1

            except multiprocessing.queues.Empty:
                all_stopped = all(not p.is_alive() for p in processes)
                if all_stopped and active_workers > 0:
                    print("Main: All worker processes stopped unexpectedly.")
                    active_workers = 0
                continue

    except KeyboardInterrupt:
        print("\nMain: Ctrl+C detected. Stopping workers...")

    finally:
        print("Main: Terminating remaining worker processes...")
        for p in processes:
            if p.is_alive():
                p.terminate()
                p.join(timeout=5)
                if p.is_alive():
                    p.kill()
                    print(f"Main: Worker {p.pid} forcefully killed.")
        print("Main: All workers stopped.")

        runtime = time.time() - start_time_main
        print("\n" + "="*20 + " Run Summary " + "="*20)
        if found_key_info:
            print("Status: SUCCESS - Target Address Found!")
            print(f"Address: {found_key_info['address']}")
            print(f"Private Key: {hex(found_key_info['key'])}")
            print(f"Check '{CRACKED_FILE}' for details.")
        else:
            print("Status: STOPPED - Target address not found in this run.")
            final_best_key, final_best_bits = load_best_key()
            print(f"Best key found overall (closest match):")
            print(f"  Bits off: {final_best_bits}")
            print(f"  Key (hex): {hex(final_best_key)}")
            print(f"  (Check '{BEST_KEY_FILE}')")
        print(f"Total Runtime: {runtime:.2f} seconds")
        print("="*53)

if __name__ == "__main__":
    if TARGET_ADDRESSES:
        main_coordinator(TARGET_ADDRESSES)
    else:
        print("Exiting because no target addresses were loaded.")

Thu Apr  3 00:57:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |     202MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Saving btc_addresses.txt to btc_addresses (10).txt
Uploaded file: btc_addresses (10).txt
Loaded 998 target addresses into memory.


ValueError: unsupported format character 'l' (0x6c) at index 47